# FreshCheck-Agent v2: entrenamiento en Google Colab (GPU)

Ejecuta el ajuste fino del Hito 3 en una GPU gratuita. **Antes de empezar:** Entorno de ejecución → Cambiar tipo → GPU (T4).

Necesita subir a su Google Drive el archivo `Original Image.zip` de FruitVision (https://data.mendeley.com/datasets/xkbjx8959c/2; licencia CC BY-NC-ND 4.0, uso académico, no redistribuir).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/FreshCheck/Original Image.zip'   # ajuste la ruta si es distinta
import os; assert os.path.exists(ZIP), 'No se encontró el zip en Drive'

In [ ]:
!git clone https://github.com/PedroC1213/FreshCheck-Agent.git
%cd FreshCheck-Agent
!git log --oneline -3

Colab ya trae torch, numpy, pandas y scikit-learn. Se usan esas versiones (quedan registradas en `entorno.json`); solo se agregan las faltantes.

In [ ]:
!pip install -q joblib scipy matplotlib pillow
import torch, sklearn, numpy, pandas
print('torch', torch.__version__, '| GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'Active la GPU: Entorno de ejecución → Cambiar tipo de entorno'

In [ ]:
!unzip -q "$ZIP" -d /content/FruitVision
!python -m src.import_fruitvision --origen /content/FruitVision

Se usan el `data/manifest.csv` y `data/splits.csv` **del repositorio** (partición idéntica a la del Hito 2). Compruebe que el hash coincide:

In [ ]:
import hashlib, json
h = hashlib.sha256(open('data/splits.csv','rb').read()).hexdigest()
h2 = json.load(open('experiments/baseline/entorno.json'))['sha256_splits']
print('splits.csv :', h[:16]); print('Hito 2     :', h2[:16])
print('COINCIDEN' if h == h2 else '¡NO COINCIDEN! no continúe sin revisar')
import pandas as pd
m = pd.read_csv('data/manifest.csv')
faltan = [i for i in m['id'] if not os.path.exists('data/raw/'+i)]
print('imágenes del manifiesto que faltan en data/raw:', len(faltan))

## 1. Caché de imágenes y línea base final

In [ ]:
!python -m src.cache
!python -m src.features
!python -m src.baseline_final

## 2. Búsqueda de hiperparámetros (solo validación; la prueba no se toca)

In [ ]:
!python -m src.tune --epocas 6

## 3. Corridas finales (3 semillas, evaluación única en prueba) y fruta no vista

In [ ]:
!python -m src.run_final

In [ ]:
!python -m src.evaluar_final

## 4. Guardar los resultados (descárguelos y súbalos al repositorio con `git`)

In [ ]:
!zip -q -r resultados_hito3.zip experiments/final models_v2 docs/analisis_errores.csv
!cp resultados_hito3.zip /content/drive/MyDrive/FreshCheck/
from google.colab import files
files.download('resultados_hito3.zip')